In [1]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.preprocessing

from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, AdaBoostRegressor, VotingRegressor
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, FunctionTransformer
from sklearn.inspection import partial_dependence, PartialDependenceDisplay, permutation_importance
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error, root_mean_squared_error
import lime
import lime.lime_tabular
import PyALE
np.random.seed(42)

In [ ]:
def datecoder():
    pass

In [2]:
df_tr = pd.read_csv('../../data/comp1/period_1_train_data.csv')
df_ts = pd.read_csv('../../data/comp1/test_x.csv')
df_tr.head(5)

,region_name_cat,district_cat,corpus_cat,developer_cat,agreement_date,floor,square,rooms_4,location_logs_count_mean,location_depth,...,location_public_transport_platform_w_mean_distance,location_water_w_mean_distance,location_university_w_mean_distance,location_leisure_w_mean_distance,location_pop_shop_cnt,price_target,hc_name_cat,interior_cat,class_cat,stage_cat
0,Город,45,538,18,2012-08-10,3.0,62.23,2,22.550466,13.0,...,0.910028,0.782675,-999.000000,0.820073,16.0,28417.424671,50,49786.0,27353,7983
1,Пригород,48,432,63,2013-05-19,11.0,22.52,студия,22.581858,13.0,...,0.902510,0.902673,-999.000000,0.990908,18.0,16728.215463,293,49786.0,97865,70661
2,Город,44,2372,126,2012-12-12,3.0,38.17,1,20.191250,13.0,...,0.851637,-999.000000,-999.000000,0.945618,7.0,18311.834458,284,49786.0,97865,70661
3,Город,14,1053,121,2012-12-10,10.0,57.48,2,23.286900,13.0,...,0.913797,1.028386,0.300026,0.828147,5.0,25171.489968,325,0.0,97865,12638
4,Город,63,2426,69,2012-02-12,3.0,41.43,1,20.599150,13.0,...,1.051049,-999.000000,-999.000000,0.991506,4.0,27324.795343,182,49786.0,97865,70661


In [3]:
# ПредобработОчка
region_map = {
    "Город": 2,
    "Пригород": 1,
    "Область": 0
}

rooms_map = {
    "студия": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    ">=4": 4
}
LogScaler = FunctionTransformer(np.log1p)
remade_dss = []
for dataset in (df_tr, df_ts):
    num_only_data = dataset.copy()
    num_only_data['region_name_cat'] = num_only_data['region_name_cat'].map(region_map)
    num_only_data['rooms_4'] = num_only_data['rooms_4'].map(rooms_map)
    if 'price_target' in num_only_data.columns.to_list():
        num_only_data['price_target'] = LogScaler.transform(num_only_data['price_target'])
    
    num_only_data.rename(columns={'region_name_cat': 'region_name'}, inplace=True)
    
    #num_only_data['agreement_date'] = pd.to_datetime(num_only_data['agreement_date'])
    #num_only_data.drop('agreement_date', axis=1, inplace=True)
    
    remade_dss.append(num_only_data)

In [4]:
cat_cols = ['agreement_date']
for col in remade_dss[0].columns:
        if '_cat' in col:
            cat_cols.append(col)
num_cols = remade_dss[0].columns.tolist()
num_cols = [col for col in num_cols if col not in cat_cols]
num_cols.remove('price_target')

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor, RadiusNeighborsRegressor
from sklearn.linear_model import QuantileRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.metrics import make_scorer
import catboost as cb

def safe_mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8)))

mape_scorer = make_scorer(safe_mape, greater_is_better=False)

trainData = remade_dss[0]
testData = remade_dss[1]

y = trainData['price_target']
X = trainData.drop('price_target', axis = 1)
X_test = testData.drop('price_target', axis=1, errors='ignore')

numFeatures = num_cols
catFeatures = cat_cols

In [6]:
numTransformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

catTransformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    #('encoder', OneHotEncoder(sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numTransformer, numFeatures),
    ('cat', catTransformer, catFeatures)
])
cat_preprocessor = ColumnTransformer([
    ('num', numTransformer, numFeatures),
])

In [8]:
# Игра с деревьями и соседями
models = {
    #"Древо принятия решений": DecisionTreeRegressor(max_features=0.65),
    "К-ближайших соседей": KNeighborsRegressor(),
    #"Случайный лес": RandomForestRegressor(max_features=0.65),
    #"Соседи в радиусе": RadiusNeighborsRegressor(),
    #"Абсолютно случайный лес": ExtraTreesRegressor()
}

param_grids = {
    "Древо принятия решений": {
        'regressor__max_depth': [20, 50, None],
        'regressor__min_samples_split': [5, 10, 20],
        'regressor__min_samples_leaf': [5, 10],
        #'regressor__criterion': ['squared_error', 'absolute_error'],
    },
    "К-ближайших соседей": {
        'regressor__n_neighbors': [3, 5, 7, 9, 15, 30, 60, 120],
        'regressor__weights': ['uniform', 'distance'],
        'regressor__p': [1, 2]
    },
    "Случайный лес": {
        'regressor__n_estimators': [120, 250],
        #'regressor__min_samples_split': [2, 5, 10, 20],
        #'regressor__criterion': ['squared_error', 'absolute_error'],
        'regressor__max_depth': [20, 50],
    },
    #"Соседи в радиусе": {
    #    'regressor__radius': [1.0, 0.5, 2.0, 3.0],
    #    'regressor__weights': ['uniform', 'distance'],
    #    'regressor__metric': ['minkowski', 'euclidean', 'manhattan', 'haversine', 'cosine'],
    #},
    "Абсолютно случайный лес": {
        'regressor__n_estimators': [120, 250],
        #'regressor__min_samples_split': [2, 5, 10, 20],
        #'regressor__criterion': ['squared_error', 'absolute_error'],
        'regressor__max_depth': [20, 50],
    }
    
}
print("Точность при перекрёстной валидации:")
best_model = None
best_score = -1

best_transformers = {}
for name, clafir in models.items():
    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('regressor', clafir)
    ])

    grid = GridSearchCV(pipeline, param_grids[name], cv=6, scoring=mape_scorer, n_jobs=-1)
    grid.fit(X, y)
    best_transformers[name+param_grids[name]] = grid.estimator
    mean_score = grid.best_score_
    
    print(f"{name:<23}: {mean_score:.6f}")
    #best_transformers[name] = grid.best_estimator_
    if mean_score > best_score:
        best_score = mean_score
        best_model = grid.best_estimator_

try:
    best_model.fit(X, y)
    final_preds = best_model.predict(X_test)

    submission = pd.DataFrame({
        'id': testData['id'] if 'id' in testData.columns else range(len(final_preds)),
        'prediction': np.expm1(final_preds)
    })
    submission.to_csv('submission.csv', index=False)
except ValueError:
    pass

Точность при перекрёстной валидации:


TypeError: can only concatenate str (not "dict") to str

Точность при перекрёстной валидации:
Древо принятия решений : -0.002513
К-ближайших соседей    : -0.003060
Случайный лес          : -0.001846
Абсолютно случайный лес: -0.001693

In [33]:
numTransformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])
# Игра с деревьями и соседями
models = {
    #"Древо принятия решений": DecisionTreeRegressor(max_features=0.65),
    #: KNeighborsRegressor(),
    #"Случайный лес": RandomForestRegressor(max_features=0.75),
    #"Соседи в радиусе": RadiusNeighborsRegressor(),
    "Абсолютно случайный лес": ExtraTreesRegressor()
}

param_grids = {
    "Древо принятия решений": {
        'regressor__max_depth': [20, 50, None],
        'regressor__min_samples_split': [5, 10, 20],
        'regressor__min_samples_leaf': [5, 10],
        #'regressor__criterion': ['squared_error', 'absolute_error'],
    },
    "К-ближайших соседей": {
        'regressor__n_neighbors': [3, 5, 7, 9, 15, 30, 60, 120],
        'regressor__weights': ['uniform', 'distance'],
        'regressor__p': [1, 2]
    },
    "Случайный лес": {
        'regressor__n_estimators': [250, 500],
        #'regressor__min_samples_split': [2, 5, 10, 20],
        #'regressor__criterion': ['squared_error', 'absolute_error'],
        'regressor__max_depth': [20, 50],
    },
    #"Соседи в радиусе": {
    #    'regressor__radius': [1.0, 0.5, 2.0, 3.0],
    #    'regressor__weights': ['uniform', 'distance'],
    #    'regressor__metric': ['minkowski', 'euclidean', 'manhattan', 'haversine', 'cosine'],
    #},
    "Абсолютно случайный лес": {
        'regressor__n_estimators': [30, 100, 250],
        #'regressor__min_samples_split': [2, 5, 10, 20],
        #'regressor__criterion': ['squared_error', 'absolute_error'],
        'regressor__max_depth': [10, 20, 30],
    }
    
}
print("Точность при перекрёстной валидации:")
best_model = None
best_score = -1

best_transformers = {}
for name, clafir in models.items():
    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('regressor', clafir)
    ])

    grid = GridSearchCV(pipeline, param_grids[name], cv=10, scoring=mape_scorer, n_jobs=-1)
    grid.fit(X, y)

    mean_score = grid.best_score_
    
    print(f"{name:<23}: {mean_score:.6f}")
    if mean_score > best_score:
        best_score = mean_score
        best_model = grid.best_estimator_

try:
    best_model.fit(X, y)
    final_preds = best_model.predict(X_test)

    submission = pd.DataFrame({
        'id': testData['id'] if 'id' in testData.columns else range(len(final_preds)),
        'prediction': np.expm1(final_preds)
    })
    submission.to_csv('submission.csv', index=False)
except ValueError:
    pass

Точность при перекрёстной валидации:
Случайный лес          : -0.001794
Абсолютно случайный лес: -0.001660


In [34]:
for es in best_transformers:
    print(best_transformers[es].steps[-1][1])

RandomForestRegressor(max_depth=50, max_features=0.75, n_estimators=500)
ExtraTreesRegressor(max_depth=50, n_estimators=300)


In [25]:
best_transformers.pop('К-ближайших соседей')
best_transformers.pop('Древо принятия решений')
a = 1

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['region_name', 'floor',
                                                   'square', 'rooms_4',
                                                   'location_logs_count_mean',
                                                   'location_depth',
                                                   'location_logs_count_std',
                                                   'location_flash_mean_mean',
                                                   'location_hds_ratio_mean_mean',
                                                   'location_hotel_w...
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))]),
                                                  ['agreement_date',
                                                   'district_cat', 'corpus_cat',
                                                   'developer_cat',
                                                   'hc_name_cat',
                                                   'interior_cat', 'class_cat',
                                                   'stage_cat'])])),
                ('regressor',
                 DecisionTreeRegressor(max_features=0.65, min_samples_leaf=5,
                                       min_samples_split=10))])

In [35]:
voting_estimators = []
for name, model in best_transformers.items():
    voting_estimators.append((name, model))
    
voting_regressor = VotingRegressor(
    estimators=voting_estimators,
    weights=None
)

voting_regressor.fit(X, y)

voting_scores = cross_val_score(voting_regressor, X, y, cv=6, scoring=mape_scorer, n_jobs=-1)
voting_mean_score = voting_scores.mean()
print(f"\nVoting Regressor: {voting_mean_score:.6f}")

final_preds = voting_regressor.predict(X_test)

submission = pd.DataFrame({
    'id': testData['id'] if 'id' in testData.columns else range(len(final_preds)),
    'prediction': np.expm1(final_preds)
})
submission.to_csv('submission1.csv', index=False)


Voting Regressor: -0.001735
